# medgraph review notebook

A guided check of the work so far. It runs the real pipeline on **synthetic Synthea data**:

1. **Phase 0.** The cohorts are pinned, intact and reproducible.
2. **Phase 1a.** One patient's FHIR bundle is read into the typed record model, and any value can be traced back to the resource it came from.
3. **Phase 1b.** That patient's labs are normalized into canonical units, and the parsers refuse to guess on ambiguous input.
4. **Cohort-wide.** Normalization outcomes and data problems across many patients.
5. **Phase 1c.** Synthetic lab reports (text and PDF, Italian and English) with exact ground truth.
6. **Phase 1d.** Extraction from those reports: the local LLM against a rule-based baseline.
7. **Phase 1e.** Hardening, checked on a fresh report set: wrong-field values, guessed dates and separators, and the eGFR equation.
8. **Phase 2.** The patient graph: its links and where each comes from, a filed report counted once, the timeline and the interactive page.
9. **Phase 3.** Guideline rules and follow-up flags: sources, one flagged patient, the cohort.
10. **Phase 4a.** The knowledge store, every cited quote checked against its stored document, retrieval, and the medication rules.
11. **Tests.** The unit test suite, run from here.
12. **What to review** at this checkpoint.

**How to run.**
- Run `make setup` once (it installs the notebook kernel).
- Open this notebook in VS Code, pick the `.venv` kernel (Python 3.12), then **Run All**. It takes about a minute, and the T7 must be mounted.
- From a terminal, `make review` re-runs it in place.

**Privacy.** Use this notebook on synthetic data only. Outputs are stripped on commit (the nbstripout hook), but real records or MIMIC must never be opened here.

In [ ]:
import json
import subprocess
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from medgraph.ingest.fhir import BundleError, read_bundle_file
from medgraph.ingest.files import content_digest, iter_data_files, sha256_file
from medgraph.normalize.analytes import ANALYTES, BY_KEY
from medgraph.normalize.labs import normalize_labs
from medgraph.normalize.numbers import parse_number
from medgraph.normalize.ranges import parse_reference_range
from medgraph.normalize.time import parse_fhir_time, parse_report_date
from medgraph.normalize.units import to_ucum
from medgraph.settings import Settings

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "scripts"))  # report and evaluation scripts
settings = Settings(_env_file=REPO / ".env")
COHORTS = settings.synthea_dir
assert COHORTS.is_dir(), f"{COHORTS} not found: is the T7 mounted?"
DEV = COHORTS / "dev-1000"
LOINC = "http://loinc.org"

pd.set_option("display.max_colwidth", 90)
# Chart style: validated reference palette, recessive grid and axes, text in ink colors.
BLUE, ORANGE, CRITICAL = "#2a78d6", "#eb6834", "#d03b3b"
SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#c3c2b7",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "text.color": INK,
        "font.size": 10,
        "lines.linewidth": 2,
        "legend.frameon": False,
        "figure.dpi": 110,
    }
)
print(f"data folder: {settings.data_dir}")

## 1. Phase 0: pinned, intact, reproducible cohorts

`scripts/generate_synthea.py` writes a `MANIFEST.json` into every cohort folder. It records the exact command, the Synthea release (checked by SHA-256), the seeds, the fixed simulation dates, the JVM timezone, and a hash of every output file.

In [ ]:
rows = []
for manifest_path in sorted(COHORTS.glob("*/MANIFEST.json")):
    m = json.loads(manifest_path.read_text())
    cmd, out = m["command"], m["outputs"]
    rows.append(
        {
            "cohort folder": manifest_path.parent.name,
            "Synthea": m["config"]["synthea"]["version"],
            "seed / clinician seed": f"{m['config']['seed']} / {m['config']['clinician_seed']}",
            "-r / -e dates": f"{cmd[cmd.index('-r') + 1]} / {cmd[cmd.index('-e') + 1]}",
            "JVM timezone": next(a.split("=")[1] for a in cmd if a.startswith("-Duser.timezone")),
            "patients (deceased)": f"{out['n_patients']:,} ({out['n_deceased']})",
            "size (MB)": round(out["total_bytes"] / 1e6),
            "seconds": m["duration_s"],
            "content digest": out["content_digest"][:16] + "…",
        }
    )
pd.DataFrame(rows).set_index("cohort folder")

**How to read it.** `pilot-10` and `pilot-10-repeat` were generated separately; the second run pretended to be on a machine in New York (`TZ=America/New_York`). Identical content digests mean byte-identical output, so a run doesn't depend on when or where it happens.

Next, re-hash the files and compare them with the manifest. This detects corruption or accidental edits on the T7.

In [ ]:
VERIFY_DEV = False  # True also re-hashes all 4.2 GB of dev-1000 (about a minute)


def verify(cohort: Path) -> str:
    manifest = json.loads((cohort / "MANIFEST.json").read_text())["outputs"]
    fhir = cohort / "fhir"
    hashes = {
        p.relative_to(fhir).as_posix(): sha256_file(p) for p in iter_data_files(fhir, "*.json")
    }
    intact = hashes == manifest["files"] and content_digest(hashes) == manifest["content_digest"]
    return f"{cohort.name}: {len(hashes)} files, " + (
        "all match the manifest" if intact else "MISMATCH"
    )


for cohort in [COHORTS / "pilot-10", *([DEV] if VERIFY_DEV else [])]:
    print(verify(cohort))

The profile's findings (`docs/data/README.md`) are properties of the Synthea generator, and they shaped Phase 1:

In [ ]:
notes_md = (REPO / "docs/data/README.md").read_text()
display(Markdown(notes_md[notes_md.index("## What the dev-1000 profile shows") :]))

## 2. Phase 1a: one patient, from FHIR bundle to typed record

Pick the first dev-1000 patient with chronic kidney disease stage 3, and read their bundle with `ingest.fhir.read_bundle_file`.

In [ ]:
CKD3 = b'"433144002"'  # SNOMED CT: chronic kidney disease stage 3
bundle_path = next(p for p in iter_data_files(DEV / "fhir", "*.json") if CKD3 in p.read_bytes())
record = read_bundle_file(bundle_path)
patient = record.patient
born = patient.birth_date.date if patient.birth_date else "?"
died = patient.deceased.date if patient.deceased else "no"
print(f"record source: {record.source[:23]}…  (SHA-256 of the bundle file)")
print(f"patient {patient.id[:8]}…: {patient.gender}, born {born}, deceased: {died}")
print(
    "not read (out of scope):", ", ".join(f"{k} {v}" for k, v in record.skipped_resources.items())
)
print("ingestion issues:", dict(Counter(i.code for i in record.issues)) or "none")
pd.Series(
    {
        "encounters": len(record.encounters),
        "conditions": len(record.conditions),
        "medication requests": len(record.medication_requests),
        "observations": len(record.observations),
        "procedures": len(record.procedures),
        "diagnostic reports": len(record.diagnostic_reports),
        "notes (deduplicated)": len(record.notes),
    },
    name="records read",
).to_frame()

### Kidney-, anemia- and diabetes-related conditions

Notice how CKD stages pile up. Synthea doesn't close an earlier stage when it records the next one, so each condition's clinical status and abatement date are kept.

In [ ]:
TERMS = ("kidney", "renal", "anemia", "diabet", "albuminuria", "proteinuria")
pd.DataFrame(
    [
        {
            "onset": c.onset.date if c.onset else None,
            "condition": c.code.label,
            "SNOMED CT": c.code.code("http://snomed.info/sct"),
            "clinical status": c.clinical_status,
            "abated": c.abatement.date if c.abatement else "",
        }
        for c in record.conditions
        if any(t in c.code.label.lower() for t in TERMS)
    ]
).sort_values("onset", ignore_index=True)

### Every value is traceable

Take one creatinine observation. The typed record keeps a `SourceRef`. Following it back into the bundle finds the exact FHIR resource it came from, with the same value, unit and time. Note that the value keeps the digits exactly as written.

In [ ]:
obs = next(o for o in record.observations if o.code.code(LOINC) in ("38483-4", "2160-0"))
q = obs.value_quantity
print(f"typed record: {obs.code.label} = {q.value!r} {q.unit} at {obs.effective.instant}")
print(f"source ref:   {obs.source}")
raw = json.loads(bundle_path.read_bytes())
original = next(
    e["resource"] for e in raw["entry"] if e["resource"].get("id") == obs.source.resource_id
)
keep = ("resourceType", "id", "effectiveDateTime", "valueQuantity")
print("raw FHIR resource:", json.dumps({k: original[k] for k in keep}, indent=2), sep="\n")

### Notes

Synthea writes each clinical note twice, once as a DocumentReference and once as a DiagnosticReport. The parser merges identical text from the same encounter into one note that lists both sources.

In [ ]:
note = record.notes[-1]
print(
    f"{len(record.notes)} notes; the latest has sources:", [s.resource_type for s in note.sources]
)
print("-" * 60)
print(note.text[:900])

## 3. Phase 1b: the same patient's labs, normalized

`normalize.labs.normalize_labs` maps in-scope LOINC codes to the analyte registry and converts each value to the analyte's canonical unit. The original value and unit are kept alongside, and `status` says whether a rule may use the value: only `ok` values can.

In [ ]:
results = normalize_labs(record)
labs = pd.DataFrame(
    [
        {
            "date": r.effective.date if r.effective else None,
            "analyte": r.analyte,
            "method": r.method,
            "as recorded": (
                f"{r.original.value} {r.original.code or r.original.unit}"
                if r.original
                else r.qualitative
            ),
            "canonical": f"{r.value:.3g} {r.unit}" if r.value is not None else "",
            "status": r.status,
            "detail": r.detail or "",
        }
        for r in results
    ]
)
display(
    labs.pivot_table(
        index="analyte", columns="status", values="date", aggfunc="count", fill_value=0
    )
)
labs.sort_values("date").tail(12)

In [ ]:
def points(analyte: str, status: str) -> list[tuple]:
    return sorted(
        (r.effective.date, float(r.value))
        for r in results
        if r.analyte == analyte and r.status == status and r.value is not None and r.effective
    )


fig, axes = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
panels = (
    ("creatinine", "Creatinine", "mg/dL"),
    ("egfr", "eGFR as reported (MDRD)", "mL/min/1.73 m²"),
)
for ax, (analyte, title, unit) in zip(axes, panels, strict=True):
    usable = points(analyte, "ok")
    if usable:
        # Dots, not a line: measurements are irregular and cluster during hospital stays.
        ax.plot(
            *zip(*usable, strict=True),
            linestyle="none",
            color=BLUE,
            marker="o",
            markersize=5,
            markeredgecolor=SURFACE,
            markeredgewidth=1,
        )
    excluded = points(analyte, "implausible")
    if excluded:
        ax.plot(
            *zip(*excluded, strict=True),
            linestyle="none",
            marker="X",
            markersize=7,
            color=CRITICAL,
            label="outside sanity bounds, not used",
        )
        ax.legend(loc="upper left")
    not_usable = sum(r.analyte == analyte and r.status != "ok" for r in results)
    note_text = f" · {not_usable} not usable" if not_usable else ""
    ax.set_title(f"{title}: {len(usable)} usable values{note_text}", loc="left", fontsize=10)
    ax.set_ylabel(unit)
    ax.set_ylim(bottom=0)
fig.suptitle(f"Lab history of patient {patient.id[:8]}… (synthetic)", x=0.01, ha="left")
fig.tight_layout()
plt.show()

A chart only plots usable values. eGFR recorded in `mL/min` has a different unit, so it isn't drawn on this axis; it's counted as "not usable" in the panel title.

### The analyte registry

The analyte registry defines what's in scope. Each non-trivial conversion factor cites its source, and a test enforces this. The sanity bounds are deliberately wide engineering limits for catching unit mix-ups and corrupt values. They are **not** clinical thresholds.

In [ ]:
pd.DataFrame(
    [
        {
            "analyte": a.key,
            "LOINC codes": ", ".join(a.loinc),
            "canonical unit": a.canonical_unit,
            "also converts from": ", ".join(u for u in a.to_canonical if u != a.canonical_unit)
            or "none",
            "sanity bounds": f"{a.sanity_low} to {a.sanity_high}",
            "conversion source": a.conversion_source or "unit arithmetic",
        }
        for a in ANALYTES
    ]
).set_index("analyte")

### Edge cases: the parsers refuse to guess

Edit the list and re-run the cell to try your own inputs. A rejection names its reason; it is never silently interpreted.

In [ ]:
FUNCTIONS = {
    f.__name__: f
    for f in (parse_number, parse_report_date, parse_fhir_time, to_ucum, parse_reference_range)
}
CASES = [
    ("parse_number", "1,2", "it"),
    ("parse_number", "1.234", "it"),
    ("parse_number", "7.5", "it"),
    ("parse_number", "1,234", "en"),
    ("parse_number", "1,234", None),
    ("parse_number", "< 0,5", "it"),
    ("parse_report_date", "07/03/2025", "it"),
    ("parse_report_date", "07/03/2025", "en-US"),
    ("parse_report_date", "7 marzo 2025", "it"),
    ("parse_report_date", "07/03/25", "it"),
    ("parse_fhir_time", "2025-03"),
    ("parse_fhir_time", "2025-03-07T10:20:30"),
    ("to_ucum", "µmol/L"),
    ("to_ucum", "ml/min/1,73 mq"),
    ("to_ucum", "{presence}"),
    ("parse_reference_range", "0,70 - 1,20", "it"),
    ("parse_reference_range", "M: 13-17 F: 12-15", "en"),
]


def attempt(name: str, *args: object) -> str:
    try:
        result = FUNCTIONS[name](*args)
    except ValueError as exc:
        return f"rejected: {exc}"
    return "not recognised" if result is None else str(result)


pd.DataFrame(
    [
        {"function": name, "input": ", ".join(map(repr, args)), "result": attempt(name, *args)}
        for name, *args in CASES
    ]
)

## 4. Cohort-wide: normalization outcomes and data problems

Read `N_PATIENTS` patients through the same pipeline. Set it to `None` for all 1,148 (about a minute). `make normalization` writes the full-cohort version to `docs/data/synthea-dev-1000-normalization.md`.

In [ ]:
N_PATIENTS = 300

rows, issues, days_after_death = [], Counter(), []
n = 0
for path in iter_data_files(DEV / "fhir", "*.json"):
    if N_PATIENTS is not None and n >= N_PATIENTS:
        break
    try:
        rec = read_bundle_file(path)
    except BundleError:
        continue  # hospital and practitioner bundles
    n += 1
    issues.update(i.code for i in rec.issues)
    if rec.patient.deceased:
        death = rec.patient.deceased.date
        days_after_death += [
            (e.period.start.date - death).days
            for e in rec.encounters
            if e.period.start and e.period.start.date > death
        ]
    for r in normalize_labs(rec):
        rows.append(
            {
                "analyte": r.analyte,
                "status": r.status,
                "value": float(r.value) if r.value is not None else np.nan,
                "recorded unit": (r.original.code or r.original.unit) if r.original else None,
                "recorded value": float(r.original.value) if r.original else np.nan,
            }
        )
cohort = pd.DataFrame(rows)
print(f"{n:,} patients, {len(cohort):,} in-scope lab results")
pd.crosstab(cohort["analyte"], cohort["status"], margins=True, margins_name="total")

### Why some values are set aside

- **Left:** creatinine in canonical mg/dL, on a log scale. Values above the sanity bound are kept and shown but never used by rules.
- **Right:** eGFR as recorded. Synthea uses one LOINC code with two units, and the two sets of values clearly differ. The `mL/min` values aren't adjusted to body surface area, so they're kept separate from `mL/min/1.73 m²` instead of being merged.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))

creatinine = cohort[cohort.analyte == "creatinine"]
bins = np.logspace(np.log10(0.2), np.log10(100), 40)
usable = creatinine.loc[creatinine.status == "ok", "value"]
excluded = creatinine.loc[creatinine.status == "implausible", "value"]
left.hist(
    usable,
    bins=bins,
    color=BLUE,
    edgecolor=SURFACE,
    linewidth=0.6,
    label=f"usable ({len(usable):,})",
)
if len(excluded):  # too few to see as bars: draw each one as a tick on the baseline
    left.scatter(
        excluded,
        [0.03] * len(excluded),
        transform=left.get_xaxis_transform(),
        marker="|",
        s=150,
        linewidths=2,
        color=CRITICAL,
        label=f"outside sanity bounds, not used ({len(excluded):,})",
    )
bound = float(BY_KEY["creatinine"].sanity_high)
left.axvline(bound, color=INK_2, linewidth=1, linestyle="--")
left.annotate(
    f"sanity bound {bound:g}",
    (bound, 0.97),
    xycoords=("data", "axes fraction"),
    xytext=(-4, 0),
    textcoords="offset points",
    ha="right",
    va="top",
    fontsize=8,
    color=INK_2,
)
left.set_xscale("log")
left.set_xlabel("creatinine, mg/dL (log scale)")
left.set_ylabel("lab results")
left.set_title("Creatinine", loc="left")
left.legend(loc="upper left")

egfr = cohort[cohort.analyte == "egfr"]
egfr_bins = np.linspace(0, 170, 35)
for unit, color, label in (
    ("mL/min/{1.73_m2}", BLUE, "mL/min/1.73 m², usable"),
    ("mL/min", ORANGE, "mL/min, not convertible"),
):
    values = egfr.loc[egfr["recorded unit"] == unit, "recorded value"]
    right.hist(
        values,
        bins=egfr_bins,
        histtype="step",
        linewidth=2,
        color=color,
        label=f"{label} ({len(values):,})",
    )
right.set_xlabel("eGFR value as recorded")
right.set_title("eGFR (LOINC 33914-3) by recorded unit", loc="left")
right.legend(loc="upper right")
fig.tight_layout()
plt.show()

In [ ]:
print("ingestion issues:", dict(issues) or "none")
pd.Series(days_after_death, name="days from death to the later encounter").describe().to_frame()

## 5. Phase 1c: synthetic lab reports with exact ground truth

`scripts/generate_lab_reports.py` prints dev-1000 Synthea lab values into reports using **code templates, never an LLM**. The model under test never reads text that it, or a related model, wrote. The ground truth is the exact printed string of every field, so scoring needs no fuzzy matching.

- **Formats.** Six layout families, in Italian and in English. English comes as en-GB (SI units) and en-US (conventional units). Every report exists as `.txt` and as a text-layer `.pdf`.
- **Development split:** four families and the usual analyte names. This is the only split used to tune the prompt and the rules.
- **Test split:**
  - adds two **held-out families** (`two_column`, `narrative`);
  - prints about half of the in-scope analytes under **held-out names** that development never shows.
- Patients never appear in both splits.
- The reference ranges printed on these reports are **illustrative test data, not clinical ranges**.

In [ ]:
import evaluate_extraction as ev  # from scripts/, see the setup cell
from medgraph.ingest.lab_report import Transcription, interpret
from medgraph.ingest.pdf import pdf_text

REPORTS = settings.lab_reports_dir / "reports-v1"
RUNS = settings.runs_dir / "extraction"
truths = {t.report_id: t for split in ("dev", "test") for t in ev.load_truths(REPORTS, split)}
overview = pd.DataFrame(
    [
        {
            "split": t.split,
            "family": t.family,
            "rows": len(t.rows),
            "held-out names": sum(r.name_seen is False for r in t.rows),
        }
        for t in truths.values()
    ]
)
overview.groupby(["split", "family"]).agg(
    reports=("rows", "size"), rows=("rows", "sum"), held_out_names=("held-out names", "sum")
)

### One report: its text, its PDF text layer and its ground truth

Below is a held-out layout in Italian. The PDF's text is rebuilt into lines from word positions, so the columns stay apart. The truth table is what a perfect transcription would copy, and `canonical_value` is what a rule would receive after conversion.

In [ ]:
example = next(t for t in truths.values() if t.family == "two_column" and t.language == "it")
stem = REPORTS / example.split / example.report_id
print(stem.with_suffix(".txt").read_text(encoding="utf-8"))
print("---- the same report, read back from the PDF's text layer ----\n")
print(pdf_text(stem.with_suffix(".pdf")))
FIELDS = ["analyte_text", "name_seen", "value_text", "unit_text", "range_text", "flag_text"]
pd.DataFrame([r.model_dump() for r in example.rows])[[*FIELDS, "canonical_value", "canonical_unit"]]

## 6. Phase 1d: extraction, local LLM against the rule-based baseline

**Transcription.** The LLM (qwen3.5:9b, local Ollama) or the rules copy each row's fields as verbatim strings.

**Interpretation.** Deterministic code then does everything that matters medically:
- It rejects any field not found in the report (`ungrounded`).
- It detects the locale.
- It maps the analyte name through a fixed table; an unknown name stays `unmapped`.
- It parses and converts the value with the same cited factors used for FHIR labs.

**The key metric is _end-to-end canonical value_:** the share of in-scope truth rows for which a rule would receive exactly the right number. The other metrics show where the losses happen. Intervals are bootstrapped over reports.

Only the LLM's transcriptions are saved. Interpretation and scoring rerun here with the current code.

In [ ]:
runs = {
    (p.parent.parent.name, p.parent.name): json.loads(p.read_text(encoding="utf-8"))
    for p in sorted(RUNS.glob("*/*/metrics.json"))
}
overall = pd.DataFrame(
    {
        f"{name}, {split}, {fmt}": values
        for (name, split), m in runs.items()
        for fmt, values in m["overall"].items()
    }
)
display(overall.map(lambda v: "n/a" if pd.isna(v) else f"{100 * v:.1f}%"))
pd.DataFrame(
    {
        f"{name}, {split}": {
            "commit": (m["provenance"].get("git") or {}).get("commit", "?")[:7],
            "uncommitted changes": (m["provenance"].get("git") or {}).get("dirty"),
            "code digest": m["provenance"].get("code_digest", "?")[:12],
            "model digest": (m["provenance"].get("model_digest") or "")[:12],
        }
        for (name, split), m in runs.items()
    }
).T

### By layout family

Held-out families show how well each method generalizes to layouts that nobody tuned it on. The rules were written for the development layouts, so 0% on held-out layouts is their expected failure, not a bug. The chart uses the test split once the LLM has a test run, and the development split until then.

End-to-end is split by analyte name. **Seen names** measures extraction itself. **Held-out names** measures the name table, which refuses names it hasn't been given. Both methods share that table, so their bars match there by design. Development has no held-out names, so that panel is empty for development.

In [ ]:
def by_family(metrics: dict, fmt: str = "text") -> pd.DataFrame:
    df = pd.DataFrame([s for s in metrics["scores"] if s["format"] == fmt])
    columns = ["matched", "truth_rows", "end_to_end_seen_ok", "in_scope_seen"]
    sums = df.groupby("family")[[*columns, "end_to_end_held_out_ok", "in_scope_held_out"]].sum()
    out = pd.DataFrame(
        {
            "row recall": sums.matched / sums.truth_rows,
            "end-to-end, seen names": sums.end_to_end_seen_ok / sums.in_scope_seen,
            "end-to-end, held-out names": sums.end_to_end_held_out_ok / sums.in_scope_held_out,
        }
    )
    held_out = set(df.loc[df.held_out_family, "family"])
    out.index = [f"{f}\n(held out)" if f in held_out else f for f in out.index]
    return out


SPLIT = "test" if ("qwen35-9b", "test") in runs else "dev"
METHODS = {"rules": ORANGE, "qwen35-9b": BLUE}
frames = {name: by_family(runs[(name, SPLIT)]) for name in METHODS if (name, SPLIT) in runs}
families = sorted(next(iter(frames.values())).index, key=lambda f: ("held out" in f, f))
METRICS = ["row recall", "end-to-end, seen names", "end-to-end, held-out names"]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.9), sharey=True)
x = np.arange(len(families))
width = 0.8 / len(frames)
for ax, metric in zip(axes, METRICS, strict=True):
    for i, (name, frame) in enumerate(frames.items()):
        values = 100 * frame[metric].reindex(families)  # NaN: no such rows, no bar
        bars = ax.bar(
            x + (i - (len(frames) - 1) / 2) * width, values, width, color=METHODS[name], label=name
        )
        labels = ["n/a" if np.isnan(v) else f"{v:.0f}" for v in values]
        ax.bar_label(bars, labels=labels, fontsize=7, color=INK_2, padding=2)
    ax.set_xticks(x, families, fontsize=8)
    ax.set_title(metric, loc="left")
    ax.set_ylim(0, 108)
axes[0].set_ylabel(f"% of rows ({SPLIT} split, text)")
axes[2].legend(*axes[0].get_legend_handles_labels(), loc="upper right")
plt.tight_layout()

### Inside one LLM transcription

The table shows the model's verbatim fields, what code made of them, and the printed truth. Set `REPORT_ID` to any saved report, for example `"test-narrative-it-00"`, and re-run the cell. An `ungrounded` row contains text that isn't in the report: it is rejected and never used.

In [ ]:
REPORT_ID = None  # None: the first saved LLM transcription
# The current runs only (an archived, superseded prompt's runs sit next to them).
saved = [
    p
    for split in ("dev", "test")
    for p in sorted((RUNS / "qwen35-9b" / split / "predictions" / "text").glob("*.json"))
    if not p.name.startswith("._")
]
path = next(p for p in saved if REPORT_ID in (None, p.stem))
record = json.loads(path.read_text(encoding="utf-8"))
truth = truths[record["report_id"]]
text = ev.report_text(REPORTS, truth, "text")
transcription = Transcription.model_validate(record["transcription"])
result = interpret(transcription, text, f"report:{truth.report_id}:text")
print(text)
seconds = record["llm"]["seconds"] if record["llm"] else float("nan")
print(
    f"{truth.report_id}: {len(transcription.rows)} rows in {seconds:.0f}s; collection date "
    f"{result.collection_date}, numbers read as {result.number_locale}"
)
matches, extra = ev.match_rows(truth, transcription)
truth_of = {m.predicted: truth.rows[m.truth] for m in matches if m.predicted is not None}
pd.DataFrame(
    [
        {
            "name (transcribed)": row.analyte_text,
            "value": row.value_text,
            "unit": row.unit_text,
            "status": row.status,
            "canonical": "" if row.value is None else f"{row.value:.4g} {row.unit}",
            "printed value (truth)": truth_of[j].value_text if j in truth_of else "(no such row)",
        }
        for j, row in enumerate(result.rows)
    ]
)

### Every difference from the truth

This lists every row the LLM missed, invented, or copied with a different field, using the same row matching as the metrics (`ev.match_rows`). Code repairs some differences downstream: a unit copied into the value field is split off by `split_value_unit`. A difference here therefore doesn't always mean a wrong number reached a rule. The end-to-end metric counts that.

In [ ]:
diffs = []
for path in saved:
    record = json.loads(path.read_text(encoding="utf-8"))
    truth = truths[record["report_id"]]
    transcription = Transcription.model_validate(record["transcription"])
    matches, extra = ev.match_rows(truth, transcription)
    where = {"report": truth.report_id, "split": truth.split}
    for m in matches:
        t = truth.rows[m.truth]
        if m.predicted is None:
            diffs.append(
                {**where, "row": t.analyte_text, "problem": "missed", "printed": t.value_text}
            )
            continue
        p = transcription.rows[m.predicted]
        for field, ok, printed, copied in (
            ("value", m.value_ok, t.value_text, p.value),
            ("unit", m.unit_ok, t.unit_text, p.unit),
            ("range", m.range_ok, t.range_text, p.reference_range),
            ("flag", m.flag_ok, t.flag_text, p.flag),
        ):
            if not ok:
                diffs.append(
                    {
                        **where,
                        "row": t.analyte_text,
                        "problem": f"{field} differs",
                        "printed": printed,
                        "transcribed": copied,
                    }
                )
    for j in extra:
        p = transcription.rows[j]
        diffs.append(
            {**where, "row": p.analyte, "problem": "not in the truth", "transcribed": p.value}
        )
errors = pd.DataFrame(
    diffs, columns=["report", "split", "row", "problem", "printed", "transcribed"]
)
print(f"{len(errors)} differences in {len(saved)} saved LLM transcriptions (text format)")
display(errors.groupby(["split", "problem"]).size().rename("rows").to_frame())
errors

## 7. Phase 1e: hardening report interpretation (ADR 0004)

Three changes decide what a report row is before it reaches the Phase 2 graph:

- **Row order.** A grounded value must still read like the row's value: after the analyte name, with no reference-range marker in between, and with its unit and range after it. Otherwise the row is `misplaced`. This catches `< 200` copied from `(rif. < 200)`, which grounding alone cannot see.
- **Evidence, not guesses.** The decimal separator and the order of day and month come from proof in the report itself. A value like `1.200`, or a date like `03/04/2025`, that nothing in the report settles is refused, not guessed. You chose this behaviour.
- **eGFR equation** on report rows, taken from the printed name.

These were developed on reports-v1, which is now development data, and then checked once on **reports-v2**:

- reports-v2 is fresh and test-only: 84 reports from patients not used in reports-v1.
- It includes a new held-out layout, `vertical`.
- The prompt is unchanged.

The overall table in section 6 already lists the `*-r2` runs.

In [ ]:
from medgraph.ingest.lab_report import report_conventions
from medgraph.normalize.analyte_names import name_key

# Every row rejected as misplaced, in the current LLM runs (reports-v1 and reports-v2).
SETS = {"qwen35-9b": "reports-v1", "qwen35-9b-r2": "reports-v2"}
misplaced, undated = [], []
for run, report_set in SETS.items():
    set_dir = settings.lab_reports_dir / report_set
    for split in ("dev", "test"):
        split_truths = (
            {t.report_id: t for t in ev.load_truths(set_dir, split)}
            if (set_dir / split).exists()
            else {}
        )
        for fmt in ("text", "pdf"):
            folder = RUNS / run / split / "predictions" / fmt
            for path in sorted(folder.glob("*.json")) if folder.exists() else []:
                if path.name.startswith("._"):
                    continue
                record = json.loads(path.read_text(encoding="utf-8"))
                truth = split_truths[record["report_id"]]
                text = ev.report_text(set_dir, truth, fmt)
                result = interpret(
                    Transcription.model_validate(record["transcription"]), text, path.stem
                )
                where = {"set": report_set, "report": truth.report_id, "format": fmt}
                for row in result.rows:
                    if row.status == "misplaced":
                        printed = next(
                            (
                                r
                                for r in truth.rows
                                if name_key(r.analyte_text) == name_key(row.analyte_text)
                            ),
                            None,
                        )
                        misplaced.append(
                            {
                                **where,
                                "row": row.analyte_text,
                                "value copied": row.value_text,
                                "printed value": printed.value_text if printed else "(no such row)",
                            }
                        )
                if result.collection_date is None:
                    undated.append(
                        {
                            **where,
                            "printed date": truth.collection_date_text,
                            "issues": "; ".join(result.issues),
                        }
                    )
print(f"{len(misplaced)} rows rejected as misplaced")
display(pd.DataFrame(misplaced))
print(f"{len(undated)} reports without a stored collection date (refused, never guessed)")
pd.DataFrame(undated)

### Try the evidence rules

Edit the texts and re-run. `report_conventions` returns the language, the decimal separator the report proves (`None` if nothing proves one), the date order it proves, and any contradictions.

In [ ]:
EXAMPLES = [
    "Collection date: 03/04/2025\nFerritin  1.200  ng/mL",  # nothing proven: both refused
    "Collected: 03/04/2025   Report date: 03/19/2025\nHGB  13.5  g/dL",  # 19 proves month-first
    "Data prelievo: 03/04/2025 ore 08.30\nCreatinina 1,32 mg/dL",  # Italian; the time is skipped
    "Collected 03/04/2025\nGlucose 5,4 mmol/L\nHGB 13.5 g/dL",  # contradiction
]
rows = []
for t in EXAMPLES:
    language, separator, order, issues = report_conventions(t)
    rows.append(
        {
            "text": t.replace("\n", " | "),
            "language": language,
            "decimal separator": separator or "not proven",
            "date order": order or "not proven",
            "issues": "; ".join(issues),
        }
    )
pd.DataFrame(rows)

## 8. Phase 2: the patient graph (ADR 0005)

Each patient's record becomes one graph. Every record is a node, including values that can't be used, and every edge says what it rests on:

- **From the data:**
  - `occurred_during`: the record's encounter reference;
  - `treated_by`: FHIR `reasonReference`, never inferred;
  - `part_of`: a report row belongs to its lab report.
- **From a cited, quoted guideline:** `monitored_by`, for example CKD to eGFR (KDIGO 2024 Practice Point 2.1.1).
- **From time order:** `precedes`.
- **From a value comparison:** `same_measurement`, a report row that repeats a recorded FHIR result, so it is counted once.

`make graphs` built all 1,148 dev-1000 patients, checked every one against the invariants in `graph/check.py`, and saved them to SQLite. The 156 synthetic lab reports are filed in their patients' records. This section reads that store; it doesn't rebuild it.

In [ ]:
from medgraph.graph.check import check_graph
from medgraph.graph.schema import edge_data, node_data
from medgraph.graph.store import GraphStore, cohort_digest
from medgraph.graph.timeline import build_timeline
from medgraph.graph.view import render_html

STORE = settings.graphs_dir / "dev-1000.sqlite"
assert STORE.exists(), f"{STORE} not found: run `make graphs`"
store = GraphStore(STORE)
meta = store.meta()
digests = store.digests()
print(f"{len(digests):,} patient graphs; store digest {meta['store_digest'][:12]}…")
print("recomputed from the stored graphs:", cohort_digest(digests) == meta["store_digest"])
kinds = pd.DataFrame(
    store.db.execute("SELECT kind, count(*) FROM nodes GROUP BY kind ORDER BY 2 DESC").fetchall(),
    columns=["node kind", "nodes"],
)
edges = pd.DataFrame(
    store.db.execute("SELECT kind, count(*) FROM edges GROUP BY kind ORDER BY 2 DESC").fetchall(),
    columns=["edge kind", "edges"],
)
display(kinds.set_index("node kind"), edges.set_index("edge kind"))

# Re-check a sample of stored graphs: loading verifies each digest, then every invariant.
sample = list(digests)[::100]
problems = {p: check_graph(store.load(p)) for p in sample}
print(f"{len(sample)} stored graphs re-checked; problems: {sum(map(len, problems.values()))}")

### One patient: links, a filed report and the timeline

The patient below has chronic kidney disease, anaemia and a filed lab report. Things to check:

- Each `monitored_by` link quotes the passage it rests on.
- Each `treated_by` link points to the record field it comes from.
- Each usable report row repeats exactly one recorded FHIR value: same analyte and day, and the value agrees at the printed precision.

In [ ]:
from view_patient import SYNTHETIC_BANNER, list_patients

PATIENT = list_patients(store, limit=1)[0].split()[0]
graph = store.load(PATIENT)
timeline = build_timeline(graph)
print(
    f"patient {PATIENT[:8]}: {graph.number_of_nodes():,} nodes, {graph.number_of_edges():,} edges"
)

monitored = [
    {
        "condition": node_data(graph, u).label,
        "analyte": node_data(graph, v).label,
        "citation": edge_data(graph, u, v, k).citation,
        "quote": edge_data(graph, u, v, k).detail,
    }
    for u, v, k in graph.edges(keys=True)
    if k == "monitored_by"
]
display(pd.DataFrame(monitored).drop_duplicates(["analyte", "citation"]))

treated = Counter(
    (node_data(graph, u).label, node_data(graph, v).label)
    for u, v, k in graph.edges(keys=True)
    if k == "treated_by"
)
display(
    pd.DataFrame(
        [{"condition": c, "treated by": t, "records": n} for (c, t), n in treated.most_common(8)]
    )
)

rows = []
for node in graph.nodes:
    d = node_data(graph, node)
    if d.kind != "report_row" or d.analyte is None:
        continue
    match = [v for _, v, k in graph.out_edges(node, keys=True) if k == "same_measurement"]
    rows.append(
        {
            "printed name": d.label,
            "as printed": d.text,
            "status": d.status,
            "usable": d.usable,
            "repeats FHIR value": ", ".join(
                f"{node_data(graph, m).original.value} {node_data(graph, m).original.unit}"
                for m in match
            )
            or "-",
        }
    )
display(pd.DataFrame(rows))
for note in timeline.notes:
    print("note:", note.text)

In [ ]:
# The usable eGFR and creatinine series; a point that a report repeats is ringed.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, analyte in zip(axes, ("egfr", "creatinine"), strict=True):
    series = next((s for s in timeline.series if s.analyte == analyte), None)
    if series is None:
        ax.set_title(f"{analyte}: no usable values", loc="left")
        continue
    times = [p.time.date for p in series.points]
    values = [float(p.value) for p in series.points]
    ax.plot(times, values, marker="o", markersize=3, color=BLUE)
    corroborated = [
        (t, v) for t, v, p in zip(times, values, series.points, strict=True) if p.corroborated_by
    ]
    if corroborated:
        ax.scatter(
            *zip(*corroborated, strict=True),
            s=90,
            facecolors="none",
            edgecolors=ORANGE,
            label="also on a filed report",
            zorder=3,
        )
        ax.legend(loc="best")
    ax.set_title(f"{series.label} ({series.unit}), {len(series.points)} usable values", loc="left")
fig.tight_layout()
plt.show()

page = settings.views_dir / "dev-1000" / f"{PATIENT}.html"
page.parent.mkdir(parents=True, exist_ok=True)
page.write_text(render_html(graph, timeline, SYNTHETIC_BANNER), encoding="utf-8")
print(f"Open the interactive page in a browser (offline, no network requests):\n{page}")

### Across the cohort: filed report rows against recorded values

Each synthetic report was printed from FHIR values. So every usable row should repeat exactly one recorded result, and none should match nothing. A row matching nothing would mean a wrong value or date got through extraction, or that matching failed. This checks the extraction evaluation (sections 6 and 7) from another angle.

In [ ]:
rows = pd.DataFrame(
    [
        {"usable": bool(a.get("usable")), "status": a.get("status")}
        for _, _, a in store.nodes_of_kind("report_row")
    ]
)
matches = Counter(
    src for (src,) in store.db.execute("SELECT src FROM edges WHERE kind = 'same_measurement'")
)
usable_rows = [(p, n) for p, n, a in store.nodes_of_kind("report_row") if a.get("usable")]
per_row = Counter(min(matches[n], 2) for _, n in usable_rows)
by_status = {k: int(n) for k, n in rows.status.value_counts().items()}
print(f"report rows: {len(rows):,}; by status: {by_status}")
print(
    f"usable rows: {len(usable_rows)}; repeating one FHIR result: {per_row[1]}; "
    f"none: {per_row[0]}; more than one: {per_row[2]}"
)

## 9. Phase 3: guideline rules and follow-up flags (ADR 0006)

The rules read each patient's graph as of an evaluation date: for Synthea, the simulation end, 2026-01-01. Your decisions:
- **CKD persistence is strict:** two abnormal values at least 90 days apart, with no normal value between.
- **For anaemia, the most recent haemoglobin decides.**
- **Medication monitoring** comes from drug labels, in Phase 4a (section 10).

Five rules produce flags:
- **CKD criteria met, no kidney diagnosis recorded** (KDIGO 2024; eGFR computed by medgraph with CKD-EPI 2021);
- **haemoglobin below the WHO 2024 cutoff, no anaemia diagnosis recorded;**
- **no GFR test, or no albuminuria test, in 12 months** for CKD (KDIGO 2024 PP 2.1.1);
- **no haemoglobin test in 12 or 6 months** for CKD stage 3, or 4-5 (KDIGO 2012 Rec 1.1.1).

Every threshold comes from a source whose passage is quoted and checked against the stored document (section 10). A test fails if a constant in the code does not appear in the quote it cites.

In [ ]:
import datetime as dt

from medgraph.rules.flags import evaluate
from medgraph.rules.model import RULESET, PatientRules
from medgraph.rules.sources import SOURCES

pd.DataFrame(
    [
        {
            "id": s.id,
            "where": s.locator,
            "quote": s.quote[:140] + ("…" if len(s.quote) > 140 else ""),
        }
        for s in SOURCES.values()
    ]
).set_index("id")

### One flagged patient

The rules run again here, live on the stored graph: a flag is derived data, recomputed whenever the rules change. The chart shows medgraph's computed eGFR next to the eGFR Synthea reports. The reported values don't follow from Synthea's own creatinine (see the cohort figures below).

In [ ]:
AS_OF = dt.date(2026, 1, 1)
stored = {
    patient: PatientRules.model_validate_json(payload)
    for patient, payload in store.db.execute(
        "SELECT patient_id, payload FROM rules WHERE ruleset = ? AND as_of = ?",
        (RULESET, AS_OF.isoformat()),
    )
}
assert stored, "run `make flags` first"
FLAGGED = next(
    p
    for p, r in sorted(stored.items())
    if any(f.rule == "ckd-criteria-no-diagnosis" for f in r.flags)
)
flagged_graph = store.load(FLAGGED)
result = evaluate(flagged_graph, AS_OF)
assert result == stored[FLAGGED]  # recomputing gives the stored result
print(f"patient {FLAGGED[:8]}: {len(result.flags)} flag(s)")
display(
    pd.DataFrame(
        [
            {"rule": a.rule, "status": a.status, "summary": a.summary, "reason": a.reason}
            for a in result.assessments
        ]
    )
)
for f in result.flags:
    print(f"\n▶ {f.title}\n  {f.statement}\n  sources: {', '.join(f.sources)}")
    for e in f.evidence:
        print(f"  evidence: {e.date} {e.value} ({e.note})")

In [ ]:
reported = next((s for s in build_timeline(flagged_graph).series if s.analyte == "egfr"), None)
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(
    [c.date for c in result.computed_egfr],
    [float(c.value) for c in result.computed_egfr],
    marker="o",
    markersize=3,
    color=BLUE,
    label="computed by medgraph (CKD-EPI 2021)",
)
if reported:
    ax.plot(
        [p.time.date for p in reported.points],
        [float(p.value) for p in reported.points],
        marker="s",
        markersize=3,
        color=MUTED,
        linestyle=":",
        label="reported by Synthea (MDRD)",
    )
ax.axhline(60, color=CRITICAL, linewidth=1, label="60: KDIGO threshold")
ax.set_ylabel("mL/min/1.73 m²")
ax.legend(loc="upper right")
ax.set_title("eGFR: what the rules use, and what the record reports", loc="left")
plt.show()

page = settings.views_dir / "dev-1000" / f"{FLAGGED}.html"
page.write_text(
    render_html(
        flagged_graph,
        build_timeline(flagged_graph),
        SYNTHETIC_BANNER,
        result.model_dump(mode="json"),
    ),
    encoding="utf-8",
)
print(f"Interactive page with the flags panel:\n{page}")

### Across the cohort

Synthea generates both the diagnoses and the values, so agreement between the criteria and Synthea's codes measures consistency with the generator, not accuracy. The rules are never tuned to it. Full tables: `docs/data/synthea-dev-1000-flags.md`.

In [ ]:
evaluated = [r for r in stored.values() if r.evaluated]
flag_counts = Counter(rule for r in evaluated for rule in {f.rule for f in r.flags})
print(f"{len(stored):,} patients, {len(evaluated):,} evaluated (deceased: not evaluated)")
display(pd.Series(flag_counts, name="patients flagged").sort_values(ascending=False).to_frame())
status = pd.DataFrame(
    [{"rule": a.rule, "status": a.status} for r in evaluated for a in r.assessments]
)
display(pd.crosstab(status["rule"], status["status"]))
report = (REPO / "docs/data/synthea-dev-1000-flags.md").read_text()
print(report.split("## Computed eGFR (CKD-EPI 2021)")[1].split("## Consistency")[0])

## 10. Phase 4a: knowledge store, retrieval index and medication rules (ADR 0007)

Your decisions: **DailyMed labels**, **hybrid retrieval with bge-m3**, and a note check that only annotates (Phase 4b).

- **Knowledge store.** Six guideline documents and reference pages, and the 201 DailyMed labels of dev-1000's prescriptions, live in `$MEDGRAPH_DATA_DIR/knowledge`. Every file is pinned by SHA-256 (`configs/knowledge/`); none is committed.
- **Quotes are checked against the stored text.** Every quote a rule, a graph link or a label statement cites is checked against the stored text, on the page or label section its locator names. This found a Phase 2 quote with an unmarked omission, and Phase 3 transcriptions that were not verbatim; all are fixed.
- **Medication rules** come only from label statements with an eGFR threshold or a test interval: metformin (eGFR below 30; 30 to 44) and epoetin alfa (haemoglobin at least monthly). Vaguer statements become notes, which are not flags.
- **Dialysis** is now recognized from procedures. The ruleset is `rules-v2`.


In [ ]:
from medgraph.rag.dailymed import LabelLock
from medgraph.rag.documents import extract_pages, load_documents
from medgraph.rag.quotes import check_quote
from medgraph.rag.spl import section_pages
from medgraph.rag.store import KnowledgeStore

KNOWLEDGE = KnowledgeStore(settings.knowledge_dir)
manifest = KNOWLEDGE.load_manifest()
docs = load_documents(REPO / "configs/knowledge/documents.yaml")
lock = LabelLock.model_validate_json(
    (REPO / "configs/knowledge/labels-dev-1000.lock.json").read_text(encoding="utf-8")
)
labels = lock.labels()
display(
    pd.DataFrame(
        [
            {
                "document": d.id,
                "licence": d.licence,
                "pinned SHA-256": d.sha256[:12],
                "stored": f"documents/{d.filename}" in manifest,
            }
            for d in docs
        ]
    ).set_index("document")
)
print(
    f"{len(lock.drugs)} RxNorm codes in dev-1000 -> {len(labels)} DailyMed labels; "
    f"{sum(d.label is None for d in lock.drugs)} codes have none"
)
display(
    pd.Series(Counter(d.tier or "no label" for d in lock.drugs), name="RxNorm codes").to_frame()
)

### Every cited quote, checked against the stored text

Each fragment between "[...]" must occur in the stored document, up to case, spacing and punctuation, on the page (or in the label section) the locator names. The last line shows that a single changed number is caught.

In [ ]:
pages = {d.id: extract_pages(KNOWLEDGE.document_path(d.filename), d) for d in docs}


def stored_pages(stored):
    if stored.startswith("label:"):
        setid = stored.removeprefix("label:").split("@")[0]
        return section_pages(KNOWLEDGE.label_path(labels[setid].path).read_bytes())
    return pages[stored]


checked = pd.DataFrame(
    [
        {
            "source": s.id,
            "stored in": s.stored.split("@")[0],
            "table transcription": bool(s.table),
            "result": "; ".join(
                check_quote(s.quote, stored_pages(s.stored), locator=s.locator, table=s.table)
            )
            or "verified",
        }
        for s in SOURCES.values()
    ]
).set_index("source")
display(checked)
altered = SOURCES["label-metformin-egfr-30"]
print(
    "metformin quote with 30 changed to 25:",
    check_quote(
        altered.quote.replace("below 30", "below 25", 1),
        stored_pages(altered.stored),
        locator=altered.locator,
    ),
)

### Retrieval: BM25 over the index, limited to a patient's sources

A search is limited to the guidelines plus the labels of the patient's own drugs, so the 15,197 label passages don't drown out the guidelines. Dense (bge-m3) and hybrid search need the local embedding model; their smoke checks, in English and Italian, are in `docs/data/knowledge-dev-1000.md`.

In [ ]:
from medgraph.rag.index import KnowledgeIndex

index = KnowledgeIndex(settings.knowledge_dir / "index-dev-1000.sqlite")
scope = {d.id for d in docs} | {"label:c3dfa8a1-d10a-4a1a-8eba-5f4e2a5a2949"}  # + metformin
for question in [
    "How often should GFR and albuminuria be assessed in people with CKD?",
    "Can metformin be used when the eGFR is below 30?",
]:
    print(f"\n{question}")
    for hit in index.bm25(question, 3, sources=scope):
        c = index.chunk(hit.chunk)
        print(f"  {hit.rank}. {c.id}  ({c.where[:70]})\n     {c.text[:200]}…")

### Medication flags and notes in the cohort

From the stored `rules-v2` results. Notes are shown on the patient page and are never counted as flags.

In [ ]:
MED_RULES = ("metformin-egfr-below-30", "metformin-egfr-30-44", "epoetin-haemoglobin-monthly")
display(
    pd.Series(
        {r: sum(any(f.rule == r for f in x.flags) for x in evaluated) for r in MED_RULES},
        name="patients flagged",
    ).to_frame()
)
display(
    pd.Series(
        Counter(rule for x in evaluated for rule in {n.rule for n in x.notes}),
        name="patients with a note",
    ).to_frame()
)
METFORMIN_PATIENT = next(
    p
    for p, r in sorted(stored.items())
    if any(f.rule == "metformin-egfr-below-30" for f in r.flags)
)
flag = next(f for f in stored[METFORMIN_PATIENT].flags if f.rule == "metformin-egfr-below-30")
print(f"patient {METFORMIN_PATIENT[:8]}\n▶ {flag.title}\n  {flag.statement}")
for e in flag.evidence:
    print(f"  evidence: {e.date} {e.label} {e.value or ''}")
source = SOURCES[flag.sources[0]]
print(f"  source: {source.citation}\n  quote: “{source.quote}”")
print(report.split("## Dialysis")[1].split("## Assessments")[0])

## 11. Test suite

This runs the unit tests (about a second). In a terminal, `make test-synthea` also parses all of dev-1000, checks that nothing in scope is dropped, and builds, checks and round-trips a sample of patient graphs.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"],
    cwd=REPO,
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout[-700:] or result.stderr[-700:])

## 12. What to review at this checkpoint

- **Analyte scope and sanity bounds** (`src/medgraph/normalize/analytes.py`). The bounds are engineering judgement, wide on purpose. For example, creatinine counts as usable up to 40 mg/dL.
- **MedicationAdministration isn't read** (17k in-hospital administrations in dev-1000). Monitoring rules will work from prescriptions.
- **Sex-specific rules** (CKD-EPI, WHO hemoglobin thresholds) will use FHIR administrative gender as a proxy, and will say so.
- **Printed reference ranges** on the synthetic reports are illustrative, not cited. medgraph never uses printed ranges as criteria.
- **Analyte identity comes from the name table, not the LLM**, so both methods score the same on held-out names. One option for later: the LLM proposes mappings for unmapped names, and a person confirms them.
- **Report eGFR rows carry the equation their printed name states** (Phase 1e), or `unspecified`. Rules recompute CKD-EPI 2021 from creatinine anyway.
- **Timings are indicative.** Seconds per report depend on what else the machine is doing.
- **Refused dates** (Phase 1e): English reports whose dates can't be ordered from the report itself have no collection date until someone confirms it (a Phase 5 workflow).
- **The row-order check assumes name, value, unit, range.** A value-first or range-first layout would have its rows rejected: recall is lost, but no wrong value gets through.
- **The monitoring table** (`src/medgraph/rules/monitoring.py`, Phase 2): CKD stages 1–4 link to eGFR, urine ACR and creatinine (KDIGO 2024), and anaemia links to haemoglobin (WHO 2024). Each link quotes the passage it rests on. End-stage renal disease, transplant and diabetic kidney disease codes are deliberately not linked yet.
- **`treated_by` comes only from `reasonReference`** (77% of medication requests, 27% of procedures in Synthea). Real records often leave it empty, and medgraph never infers a link.
- **Same measurement at the printed precision.** Two different measurements with the same analyte, day and printed value would count as one.
- **The viewer** hides concepts that have no link by default (search still finds them), and the timeline opens on the span of the lab values. Both are display choices; nothing is dropped.
- **`docs/data/` was never committed**, because the `.gitignore` pattern `data/` matched it. It is now anchored to the repository root. Commit `docs/data/`.
- **Phase 3 rules** (`src/medgraph/rules/`, ADR 0006). Check that each source quote says what the rule does with it (`rules/sources.py`), and check the code lists of matching diagnoses (`rules/diagnoses.py`).
- **Anaemia is not assessed over 65** (WHO 2024 sets no cutoff there) or under 6 months. In pregnancy it is assessed only where the trimester can't change the answer. Current smokers get WHO's +3 g/L.
- **Synthea's reported eGFR doesn't follow from its own creatinine**, so medgraph's CKD criteria disagree with Synthea's CKD codes by construction of the generator, not because of the rules.
- **Follow-up flags count any recorded test**, even one whose value could not be used. Kidney failure, dialysis and transplant are outside these rules.
- **Decisions and evidence:** `docs/decisions/0001-…` to `0006-…`; results in `docs/results/` and `docs/data/`.
- **Phase 4a: label choice** (`rag/dailymed.py`, rule `labels-v1`). For each product, medgraph takes the most recently published label of the best tier (NDA/BLA, then ANDA, then any). Some choices are old repackager copies; the knowledge report lists them.
- **Phase 4a: what becomes a flag.** Only an eGFR threshold or a test interval stated in a label: metformin and epoetin alfa. "Monitor renal function periodically" with an NSAID becomes a note. Creatinine-clearance thresholds are not used, because medgraph does not compute creatinine clearance.
- **Phase 4a: dialysis window.** A session in the 30 days before the evaluation date. This is medgraph's definition, and any window from 7 to 250 days gives the same result on dev-1000.
- **Phase 4a: corrected quotes.** A Phase 2 quote had an unmarked omission. The CKD-EPI and MDRD quotes were my transcriptions and now quote NIDDK's text. Two WHO and KDIGO tables are marked as transcriptions (ADR 0007).
